# AvertCare - ML Training Pipeline
This notebook covers Phase 1 & 2: Data Cleaning, EDA, Split, and Preprocessing.

In [ ]:
# Install dependencies (for Colab)
!pip install -q pandas numpy scikit-learn torch shap matplotlib joblib ucimlrepo

In [ ]:
import os
import sys

# Setup BASE_DIR for Colab or Local
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/AvertCare'):
        !git clone https://github.com/Avert-care/AvertCare.git /content/AvertCare
    BASE_DIR = '/content/AvertCare'
    sys.path.append(os.path.join(BASE_DIR, 'ml_engine'))
else:
    BASE_DIR = os.path.dirname(os.getcwd()) # Assuming running from notebooks/
    sys.path.append(os.path.join(BASE_DIR, 'ml_engine'))

print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from clean import clean_data
from split import split_data, build_preprocessors

print("Fetching dataset...")
try:
    dataset = fetch_ucirepo(id=296)
    X = dataset.data.features
    y = dataset.data.targets
    ids = dataset.data.ids
    df = pd.concat([ids, X, y], axis=1)
except Exception as e:
    print(f"Failed to fetch: {e}")
    # Fallback local path
    df = pd.read_csv(os.path.join(BASE_DIR, 'ml_engine', 'data', 'raw', 'diabetic_data.csv'))

print("Shape:", df.shape)

In [ ]:
print("--- PHASE 1: CLEANING ---")
df_clean = clean_data(df)
df_clean.head()

In [ ]:
print("--- PHASE 2: SPLIT & PREPROCESS ---")
df_train, df_val, df_test = split_data(df_clean)

numeric_cols = df_clean.select_dtypes(include=['number']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['encounter_id', 'patient_nbr', 'readmitted_binary']]
categorical_cols = df_clean.select_dtypes(exclude=['number']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['encounter_id', 'patient_nbr']]

prep_onehot, prep_ft = build_preprocessors(df_train, numeric_cols, categorical_cols)
print("Preprocessors built successfully!")